# Command Line Interface Overview

> In some cases we want to tune _many_ models for different response variables, species, etc. `vnn.py` is designed to make this mostly painless. 

## Design Notes:


### Organizing Ideas: 
The key ideas behind this script's design are:

1. The script should end up doing the right thing (or at least a reasonable thing) even if not told what to do. 
1. There should be several ways to control the script's behavior, providing more flexibility as the difficulty increases. 


As a result of the first point the script will begin with start with default parameters that are quick to run and look for input files when not specified. This means that a user that doesn't provide enough information to get what they want (an optimized model) doesn't wait a long time to discover this and gets something funcitonal to boot.

Per the second point, I've provided several ways to modify how this tool behaves. As these move from 'standard user' to 'power user' we assume more of the user. We also assume that options which are explicit (flags & settings files) should supercede default values. We'll explore the first 4 below. In order of customization potential these are: 

|    | Action                     | In Brief                  |
|----|----------------------------|---------------------------|
| 1. | Default `vnn.py`           | No customization. `vnn.py` runs with defaults and infers input files. |
| 2. | Command line options/flags | Flags allow for disambiguation (use _this_ gene annotation) and control of parameters likely to be modified (e.g. number of hyperparameter tuning trials).|
| 3. | JSON settings files        | More options are provided through these files. Most users will likely customize their experiments with these files.  |
| 4. | Replacement Functions      | While most flags are a subset of settings that can be provided in JSON, a few allow for replacing existing functions with custom versions. |
| 5. | Script rewrite             | As a last resort, this script can be edited directly.|


### Assume that Inconvenience Confers Intention

To help reason about what the user intends, I assume that providing options in the CLI implies more intentionality than a settings file -- If someone includes `--max_epoch 256` they want to use that not whatever is in the settings files. 

Based on this assumption, we start with default and infered parameters and then overwrite them if settings files or flags are available.  

```mermaid
flowchart LR

A[Defaults]
A --> B[Infered Parameters]  
B --> C[JSON Parameters]
C --> D[Flags]
``` 

## Default `vnn.py`

The pure default behavior is seldom what the user wants -- so the script should run fast so they don't have to wait long to discover this. 

While `vnn.py` can't read the user's mind it can give them a good starting place. In the process of running, it will look in the `pwd` for input files (phenotype, genotype, & gene models) and will try to find or download a graph structure -- and will make paths for these explicit. For reproducibiltiy settings files will be written which the user can customize to get the behavior they _actually_ want. 

## Command Line Flags

Command line flags all have a help annotation which can be examined in the terminal. Here I'll focus on the broad strokes. 

As a general rule flags are a subset of the options available through JSON files (see exceptions in "Replacement Functions"). The break down into two main categories: paths and parameters.

### Path Flags

Different experiments may use the same input data so these can be specified. These settings are a subset of `params_data.json`
* `--graph_cache_path` (where graph json files are)
* `--gff_path` (Gene model)
* `--hmp_path` (HapMap)
* `--phno_path`  (Phenotype file with a Taxa column)
* `--cache_path` (where outputs are written)
* `--model_path` (saved model dictionary ending in `.pt`)

Similarly, json settings could be explictly loaded.
* `--params_data` (defines inputs)
* `--params_run` (defines what the script will do)
* `--params` (defines one model)
* `--params_list` (defines a hyperparameter space)

### Parameter Flags

<span style="color:red">**TODO:**</span> add `species` and `num_nucleotides` flags

Parameters controlling the script's behavior are mostly a subset of `params_run.json` with the `dataloader_shuffle_*` parameters being housed in `params_data.json`. 

The most important of these is `--run_mode` which controls if the script conducts hyperparameter tuning (`tune`), model fitting (`train`) or makes predictions (`predict`).

General settings: 
* `--run_mode` (what operation should take place?)
* `--batch_size`
* `--max_epoch` 
* `--dataloader_shuffle_train` (should observations be shuffled? Set `False` for `predict`)
* `--dataloader_shuffle_valid` (should observations be shuffled? Defaults to `False`)

Hyperparameter settings:
* `--tune_trials` (How many hyperparameter combinations should be tested in this session?)
* `--tune_max` (How many hyperparameter combinations should be tested *overall*?)
* `--tune_force` (Should trials be run even if the maximum has been reached?)

Tuning settings:
* `--train_from_ax` (Should the best parameters found by the Adaptive Experimentation platform be used?)
* `--train_save` (Should the model be saved?)











## JSON Settings Files

`params_data.json` contains information about where the data is located, what holdout to use, and if the dataloader should shuffled entries. It's expected that a user will customize this.

`params_run.json` contains information about the current session's behavior. It's expected that a user will customize this either once (and then change the mode with a `--run_mode`) or multiple times.

`params.json` by default defines a *tiny* network, each node producing a single value and there being no dropout. This should be customized if the target network is known otherwise it should be determined via hyperparameter tuning. 

`params_list.json` defines the hyperparameter search space. It _probably_ shouldn't be changed. If it is though, `vnn.py` will update the default search space by matching each dictionary's `'name'` _and_ will add any entries with new names to the list.  

## Replacement Functions

Advanced control is allowed by _overwritting_ functions from `sparsevnn` the script uses.

The expected functions to be replaced are `plDNN_general`


`SparseVNN`
<span style="color:red">**TODO:**</span> Should this be replaced or should we have a way to use a custom nnet that's named something else instead?


<span style="color:red">**TODO:**</span> Manual cxn

<span style="color:red">**TODO:**</span> Overwrite functions

<span style="color:red">**TODO:**</span> undocumented feature where a bad run mode will make the setup files and exit. Maybe this should be the default

## Output Organization